# 06 - Merge Sources

## Objetivo
Integrar las fuentes reconstruidas del proyecto en un dataset único listo para análisis posterior y modelado.

## Fuentes esperadas
- `data/interim/milking.parquet`
- `data/interim/rumination.parquet`
- `data/interim/weather.parquet`
- `data/interim/pdf_events.parquet`

## Estrategia
1. Cargar cada fuente si existe.
2. Normalizar claves (`cow_id`, `date`).
3. Agregar cada fuente a nivel diario.
4. Usar `milking_daily` como base principal cuando exista.
5. Unir:
   - rumia por `cow_id + date`
   - clima por `date`
   - eventos PDF por `cow_id + date`
6. Guardar:
   - dataset completo
   - dataset filtrado al rango donde existe rumia

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Rutas del proyecto

In [ ]:
CURRENT = Path.cwd().resolve()

if (CURRENT / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError("No se encontró data/interim ni en el directorio actual ni en el padre.")

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INTERIM_DIR  :", INTERIM_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

## 2. Archivos esperados

In [ ]:
MILKING_PATH = INTERIM_DIR / "milking.parquet"
RUMINATION_PATH = INTERIM_DIR / "rumination.parquet"
WEATHER_PATH = INTERIM_DIR / "weather.parquet"
PDF_EVENTS_PATH = INTERIM_DIR / "pdf_events.parquet"

for p in [MILKING_PATH, RUMINATION_PATH, WEATHER_PATH, PDF_EVENTS_PATH]:
    print(f"{p.name:22} -> {'OK' if p.exists() else 'MISSING'}")

## 3. Funciones auxiliares

In [ ]:
def LoadParquetIfExists(path: Path):
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {path.name:22} -> {df.shape}")
        return df
    print(f"Skipped {path.name:21} -> file not found")
    return None


def NormalizeCowId(df: pd.DataFrame):
    df = df.copy()
    candidate_cols = ["cow_id", "animal_id", "vid", "resolved_vid", "resolved_cow_id"]
    for col in candidate_cols:
        if col in df.columns:
            df["cow_id"] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            return df
    return df


def EnsureDateColumn(df: pd.DataFrame, datetime_candidates, out_col="date"):
    df = df.copy()

    if out_col in df.columns:
        df[out_col] = pd.to_datetime(df[out_col], errors="coerce").dt.normalize()
        return df

    for col in datetime_candidates:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            df[out_col] = df[col].dt.normalize()
            return df

    raise KeyError(f"No se pudo crear '{out_col}'. Columnas candidatas no encontradas: {datetime_candidates}")


def ToMinutesFromHHMM(series: pd.Series):
    s = series.astype(str).str.strip()
    td = pd.to_timedelta(s, errors="coerce")
    return td.dt.total_seconds() / 60


def CoerceBooleanLikeToNumeric(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "si": 1, "sí": 1,
        "x": 1, "": np.nan,
        "nan": np.nan, "none": np.nan
    }
    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.combine_first(numeric)

## 4. Agregación diaria por fuente

In [ ]:
def DailyAggMilking(df: pd.DataFrame):
    df = df.copy()
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["hora_inicio", "timestamp", "datetime", "date"])

    # Conversión de intervalos si existen
    if "duracion_mmss" in df.columns and "duracion_min" not in df.columns:
        df["duracion_min"] = ToMinutesFromHHMM(df["duracion_mmss"])

    if "intervalo_ordeno_hhmm" in df.columns and "intervalo_ordeno_min" not in df.columns:
        df["intervalo_ordeno_min"] = ToMinutesFromHHMM(df["intervalo_ordeno_hhmm"])

    numeric_candidates = [
        "produccion_kg", "di", "dd", "ti", "td",
        "duracion_min", "intervalo_ordeno_min",
        "numero_ordeno"
    ]

    for c in numeric_candidates:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    event_like_cols = ["patada", "incompleto", "pezones_no_encontrados"]
    for c in event_like_cols:
        if c in df.columns:
            df[c] = CoerceBooleanLikeToNumeric(df[c])

    agg = {}
    if "produccion_kg" in df.columns: agg["produccion_kg"] = "sum"
    if "di" in df.columns: agg["di"] = "sum"
    if "dd" in df.columns: agg["dd"] = "sum"
    if "ti" in df.columns: agg["ti"] = "sum"
    if "td" in df.columns: agg["td"] = "sum"
    if "numero_ordeno" in df.columns: agg["numero_ordeno"] = "count"
    if "duracion_min" in df.columns: agg["duracion_min"] = "sum"
    if "intervalo_ordeno_min" in df.columns: agg["intervalo_ordeno_min"] = "mean"
    if "patada" in df.columns: agg["patada"] = "sum"
    if "incompleto" in df.columns: agg["incompleto"] = "sum"
    if "pezones_no_encontrados" in df.columns: agg["pezones_no_encontrados"] = "sum"

    for c in ["ubre", "destino_leche", "ms", "source_file", "source_sheet"]:
        if c in df.columns:
            agg[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(agg)
          .reset_index()
    )

    daily = daily.rename(columns={
        "numero_ordeno": "ordenos_dia",
        "duracion_min": "duracion_total_min",
        "intervalo_ordeno_min": "intervalo_ordeno_prom_min",
        "patada": "patadas_dia",
        "incompleto": "incompletos_dia",
        "pezones_no_encontrados": "pezones_no_encontrados_dia"
    })

    return daily


def DailyAggRumination(df: pd.DataFrame):
    df = df.copy()
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["date", "timestamp", "datetime", "fecha"])

    rename_map = {
        "ruminating_minutes": "rumia_min",
        "ruminating": "rumia_min",
        "resolved_group": "group_id",
        "days_in_milk": "days_in_milk",
        "lactation_age": "lactation_age",
        "weekday": "weekday",
        "month": "month",
        "source_file": "source_file"
    }

    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df = df.rename(columns={old: new})

    for c in ["rumia_min", "days_in_milk", "lactation_age", "weekday", "month"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = {}
    for c in ["rumia_min", "days_in_milk", "lactation_age", "weekday", "month"]:
        if c in df.columns:
            agg[c] = "mean"
    for c in ["group_id", "source_file"]:
        if c in df.columns:
            agg[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(agg)
          .reset_index()
    )

    return daily


def DailyAggWeather(df: pd.DataFrame):
    df = df.copy()
    df = EnsureDateColumn(df, ["time", "date", "datetime", "timestamp", "fecha"])

    rename_map = {
        "temperatura": "temperature_c",
        "temperature": "temperature_c",
        "temperature_2m": "temperature_c",
        "humedad": "humidity_pct",
        "humidity": "humidity_pct",
        "relative_humidity_2m": "humidity_pct",
        "lluvia": "rain_mm",
        "rain": "rain_mm",
        "precipitation": "rain_mm",
        "viento": "wind_speed",
        "wind": "wind_speed",
        "wind_speed_10m": "wind_speed",
        "pressure_msl": "pressure_msl"
    }

    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df = df.rename(columns={old: new})

    numeric_cols = [
        c for c in [
            "temperature_c",
            "humidity_pct",
            "pressure_msl",
            "rain_mm",
            "wind_speed",
            "heat_index"
        ] if c in df.columns
    ]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if numeric_cols:
        daily = df.groupby("date", dropna=False)[numeric_cols].mean().reset_index()
    else:
        daily = df[["date"]].drop_duplicates().copy()

    return daily


def DailyAggPdfEvents(df: pd.DataFrame):
    df = df.copy()
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["event_date", "date", "fecha_evento", "timestamp", "datetime"])

    if "raw_event_text" not in df.columns:
        text_col = None
        for c in ["evento", "descripcion", "event_text", "raw_text"]:
            if c in df.columns:
                text_col = c
                break
        if text_col is not None:
            df = df.rename(columns={text_col: "raw_event_text"})
        else:
            df["raw_event_text"] = ""

    df["raw_event_text"] = df["raw_event_text"].astype(str)

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(
              eventos_pdf_count=("raw_event_text", "count"),
              eventos_pdf_text=("raw_event_text", lambda s: " | ".join(s.dropna().astype(str).head(10)))
          )
          .reset_index()
    )

    return daily

## 5. Cargar fuentes

In [ ]:
milking_raw = LoadParquetIfExists(MILKING_PATH)
rumination_raw = LoadParquetIfExists(RUMINATION_PATH)
weather_raw = LoadParquetIfExists(WEATHER_PATH)
pdf_events_raw = LoadParquetIfExists(PDF_EVENTS_PATH)

## 6. Agregar a nivel diario

In [ ]:
milking_daily = DailyAggMilking(milking_raw) if milking_raw is not None else None
rumination_daily = DailyAggRumination(rumination_raw) if rumination_raw is not None else None
weather_daily = DailyAggWeather(weather_raw) if weather_raw is not None else None
pdf_events_daily = DailyAggPdfEvents(pdf_events_raw) if pdf_events_raw is not None else None

for name, df_src in [
    ("milking_daily", milking_daily),
    ("rumination_daily", rumination_daily),
    ("weather_daily", weather_daily),
    ("pdf_events_daily", pdf_events_daily),
]:
    if df_src is None:
        print(f"{name:18} -> None")
    else:
        print(f"{name:18} -> {df_src.shape}")

## 7. Vista rápida

In [ ]:
if milking_daily is not None:
    display(milking_daily.head())

if rumination_daily is not None:
    display(rumination_daily.head())

if weather_daily is not None:
    display(weather_daily.head())

if pdf_events_daily is not None:
    display(pdf_events_daily.head())

## 8. Diagnóstico de traslape de claves

In [ ]:
print("=== KEY OVERLAP DIAGNOSTIC ===")

if milking_daily is not None and rumination_daily is not None:
    milking_ids = set(milking_daily["cow_id"].dropna().unique())
    rumination_ids = set(rumination_daily["cow_id"].dropna().unique())

    print("Milking cows:", len(milking_ids))
    print("Rumination cows:", len(rumination_ids))
    print("Shared cows:", len(milking_ids & rumination_ids))

    milking_keys = set(zip(milking_daily["cow_id"], milking_daily["date"]))
    rumination_keys = set(zip(rumination_daily["cow_id"], rumination_daily["date"]))

    print("Shared cow-date keys:", len(milking_keys & rumination_keys))

if milking_daily is not None and pdf_events_daily is not None:
    milking_ids = set(milking_daily["cow_id"].dropna().unique())
    pdf_ids = set(pdf_events_daily["cow_id"].dropna().unique())

    print("\nMilking cows:", len(milking_ids))
    print("PDF cows:", len(pdf_ids))
    print("Shared cows:", len(milking_ids & pdf_ids))

    milking_keys = set(zip(milking_daily["cow_id"], milking_daily["date"]))
    pdf_keys = set(zip(pdf_events_daily["cow_id"], pdf_events_daily["date"]))

    print("Shared cow-date keys:", len(milking_keys & pdf_keys))

## 9. Rangos temporales

In [ ]:
if milking_daily is not None:
    print("Milking range   :", milking_daily["date"].min(), "->", milking_daily["date"].max())
if rumination_daily is not None:
    print("Rumination range:", rumination_daily["date"].min(), "->", rumination_daily["date"].max())
if pdf_events_daily is not None:
    print("PDF range       :", pdf_events_daily["date"].min(), "->", pdf_events_daily["date"].max())
if weather_daily is not None:
    print("Weather range   :", weather_daily["date"].min(), "->", weather_daily["date"].max())

## 10. Elegir base del merge

Se usa `milking_daily` como base principal cuando exista.

In [ ]:
if milking_daily is not None:
    merged = milking_daily.copy()
    print("Base del merge: milking_daily")
elif rumination_daily is not None:
    merged = rumination_daily.copy()
    print("Base del merge: rumination_daily")
else:
    raise ValueError("Se requiere al menos milking.parquet o rumination.parquet para construir la base del merge.")

## 11. Merge con rumia

In [ ]:
if rumination_daily is not None and merged is not rumination_daily:
    merged = merged.merge(
        rumination_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_rum")
    )

print("Shape after rumination merge:", merged.shape)
display(merged.head())

## 12. Merge con clima

In [ ]:
if weather_daily is not None:
    merged = merged.merge(
        weather_daily,
        on="date",
        how="left",
        suffixes=("", "_weather")
    )

print("Shape after weather merge:", merged.shape)
display(merged.head())

## 13. Merge con eventos PDF

In [ ]:
if pdf_events_daily is not None:
    merged = merged.merge(
        pdf_events_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_pdf")
    )

print("Shape after pdf merge:", merged.shape)
display(merged.head())

## 14. Variables temporales mínimas

In [ ]:
merged["date"] = pd.to_datetime(merged["date"], errors="coerce")
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day
merged["weekday_date"] = merged["date"].dt.weekday

## 15. Reporte de nulos del dataset completo

In [ ]:
sort_cols = [c for c in ["cow_id", "date"] if c in merged.columns]
if sort_cols:
    merged = merged.sort_values(sort_cols).reset_index(drop=True)

nan_report = pd.DataFrame({
    "column": merged.columns,
    "nan_count": merged.isna().sum().values,
    "nan_pct": (merged.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

print("Shape merged completo:", merged.shape)
display(nan_report.head(30))

## 16. Crear dataset filtrado al rango de rumia

Como la rumia tiene cobertura temporal más corta, esta versión sirve mejor para modelado cuando se quiera usar esa fuente.

In [ ]:
if rumination_daily is not None:
    rum_min = rumination_daily["date"].min()
    rum_max = rumination_daily["date"].max()

    merged_model = merged[
        (merged["date"] >= rum_min) &
        (merged["date"] <= rum_max)
    ].copy()

    print("Rango rumia:", rum_min, "->", rum_max)
else:
    merged_model = merged.copy()

print("Shape merged completo:", merged.shape)
print("Shape merged_model   :", merged_model.shape)

if "rumia_min" in merged_model.columns:
    rumia_cov = merged_model["rumia_min"].notna().mean() * 100
    print(f"Cobertura de rumia en merged_model: {rumia_cov:.2f}%")

## 17. Reporte de nulos del dataset para modelado

In [ ]:
nan_report_model = pd.DataFrame({
    "column": merged_model.columns,
    "nan_count": merged_model.isna().sum().values,
    "nan_pct": (merged_model.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

display(nan_report_model.head(30))

## 18. Validaciones rápidas

In [ ]:
print("=== VALIDACIÓN FINAL ===")
print("Merged completo:")
print("  filas:", merged.shape[0])
print("  columnas:", merged.shape[1])

if "cow_id" in merged.columns:
    print("  vacas:", merged["cow_id"].nunique(dropna=True))
print("  rango fechas:", merged["date"].min(), "->", merged["date"].max())

if "produccion_kg" in merged.columns:
    print("  producción total:", merged["produccion_kg"].sum())

if "rumia_min" in merged_model.columns:
    print("\nMerged model:")
    print("  filas:", merged_model.shape[0])
    print("  vacas:", merged_model["cow_id"].nunique(dropna=True))
    print("  rumia total:", merged_model["rumia_min"].sum(skipna=True))

## 19. Guardar resultados

In [ ]:
OUTPUT_FULL = PROCESSED_DIR / "training_dataset.parquet"
OUTPUT_FULL_CSV = PROCESSED_DIR / "training_dataset.csv"

OUTPUT_MODEL = PROCESSED_DIR / "training_dataset_with_rumination.parquet"
OUTPUT_MODEL_CSV = PROCESSED_DIR / "training_dataset_with_rumination.csv"

merged.to_parquet(OUTPUT_FULL, index=False)
merged.to_csv(OUTPUT_FULL_CSV, index=False)

merged_model.to_parquet(OUTPUT_MODEL, index=False)
merged_model.to_csv(OUTPUT_MODEL_CSV, index=False)

print("Saved:")
print(" -", OUTPUT_FULL)
print(" -", OUTPUT_FULL_CSV)
print(" -", OUTPUT_MODEL)
print(" -", OUTPUT_MODEL_CSV)

## 20. Próximo paso
Con este notebook ya puedes pasar a:
- `07_feature_engineering.ipynb`
- `08_model_training.ipynb`

La versión `training_dataset_with_rumination.parquet` es la más apropiada si el modelo necesita usar rumia.